In [7]:
from selenium.webdriver.chrome.service import Service
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd

service = Service("C:/Users/sippa/Downloads/Work/chromedriver/chromedriver.exe")
driver = webdriver.Chrome(service=service)

In [8]:
email = "forgptkung@gmail.com"
password = "ForStockscreener12"
column = "Technical Rating"

# Navigate to screener
driver.get("https://www.tradingview.com/screener/")
time.sleep(3)

# Add column
addcolumn = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, "button[data-qa-id='screener-add-column-button']"))
)
addcolumn.click()

searchcolumn = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "input[aria-label='Type column name']"))
)
searchcolumn.send_keys(column)
time.sleep(2)

technicalrating = driver.find_element(By.CLASS_NAME, "highlighted-xWsxD6Lf")
technicalrating.click()
time.sleep(1)

adddaycolumn = driver.find_element(By.CSS_SELECTOR, "button[data-overflow-tooltip-text='Add column']")
adddaycolumn.click()
time.sleep(2)

# Sign in
try:
    emailbutton = driver.find_element(By.CSS_SELECTOR, "button[name='Email']")
    emailbutton.click()
    
    usernameinput = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.ID, "id_username"))
    )
    usernameinput.send_keys(email)
    
    passwordinput = driver.find_element(By.ID, "id_password")
    passwordinput.send_keys(password)
    
    signinbutton = driver.find_element(By.CSS_SELECTOR, "button[data-overflow-tooltip-text='Sign in']")
    signinbutton.click()
    
    print("Signed in successfully!")
except Exception as e:
    print(f"Sign in error: {e}")

# Wait for page to fully load
time.sleep(5)

Signed in successfully!


In [9]:
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.actions.wheel_input import ScrollOrigin
from io import StringIO

# Wait for page to load after sign in
time.sleep(5)

# Find the scrollable container in the lower half
try:
    scrollable = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "div.shadow-zuRb9wy5"))
    )
except:
    try:
        scrollable = driver.find_element(By.CLASS_NAME, "wrap-vSb6C0Bj")
    except:
        scrollable = driver.find_element(By.TAG_NAME, "table")

print("Starting data load...")
print(f"Scrollable element found: {scrollable.tag_name}")

last_row_count = 0
no_change_count = 0
scroll_attempts = 0
loaded_stocks = set()

while scroll_attempts < 1000:
    # Scroll from the scrollable element origin (lower half)
    scroll_origin = ScrollOrigin.from_element(scrollable)
    ActionChains(driver)\
        .scroll_from_origin(scroll_origin, 0, 10000)\
        .perform()
    
    # Reduced wait time
    time.sleep(1)
    
    # Check row count every 10 scrolls (less frequent)
    if scroll_attempts % 10 == 0:
        try:
            temp_df = pd.read_html(StringIO(driver.page_source))
            current_row_count = len(temp_df[0])
            
            # Print newly loaded stock names (limit to 5)
            ticker_col = 'Ticker' if 'Ticker' in temp_df[0].columns else 'Symbol'
            if ticker_col in temp_df[0].columns:
                current_stocks = set(temp_df[0][ticker_col].tolist())
                new_stocks = current_stocks - loaded_stocks
                if new_stocks:
                    print(f"New stocks: {len(new_stocks)}")
                    loaded_stocks = current_stocks
            
            print(f"Total: {current_row_count} rows (scroll: {scroll_attempts})")
            
            # Check if loading is complete
            if current_row_count == last_row_count:
                no_change_count += 1
                if no_change_count >= 3:
                    print(f"\n✓ Complete! Total rows: {current_row_count}")
                    break
            else:
                no_change_count = 0
                last_row_count = current_row_count
                
        except Exception as e:
            print(f"Error: {e}")
    
    scroll_attempts += 1

# Get final data
data_df = pd.read_html(StringIO(driver.page_source))
print(f"\nFinal shape: {data_df[0].shape}")
data_df[0]

Starting data load...
Scrollable element found: div
New stocks: 200
Total: 200 rows (scroll: 0)
New stocks: 600
Total: 800 rows (scroll: 10)
New stocks: 500
Total: 1300 rows (scroll: 20)
New stocks: 401
Total: 1705 rows (scroll: 30)
New stocks: 400
Total: 2105 rows (scroll: 40)
New stocks: 400
Total: 2505 rows (scroll: 50)
New stocks: 400
Total: 2900 rows (scroll: 60)
New stocks: 301
Total: 3205 rows (scroll: 70)
New stocks: 400
Total: 3605 rows (scroll: 80)
New stocks: 400
Total: 4005 rows (scroll: 90)
New stocks: 400
Total: 4400 rows (scroll: 100)
New stocks: 301
Total: 4705 rows (scroll: 110)
New stocks: 400
Total: 5100 rows (scroll: 120)
Total: 5100 rows (scroll: 130)
Total: 5100 rows (scroll: 140)
Total: 5100 rows (scroll: 150)

✓ Complete! Total rows: 5100

Final shape: (5100, 14)


,Symbol,Price,Change %,Volume,Rel Volume,Market cap,P/E,EPS dilTTM,EPS dil growthTTM YoY,Div yield %TTM,Sector,Analyst Rating,Tech Rating,Unnamed: 13
0,NVDANVIDIA Corporation,186.47 USD,−0.64%,124.8 M,0.74,4.53 T USD,46.19,4.04 USD,+59.03%,0.02%,Electronic technology,Strong buy,Buy,NaN
1,GOOGAlphabet Inc.,333.59 USD,+1.57%,18.5 M,0.87,4.02 T USD,32.91,10.14 USD,+34.47%,0.25%,Technology services,Strong buy,Buy,NaN
2,AAPLApple Inc.,255.41 USD,+2.97%,55.97 M,1.12,3.75 T USD,34.24,7.46 USD,+22.89%,0.42%,Electronic technology,Buy,Sell,NaN
3,MSFTMicrosoft Corporation,470.28 USD,+0.93%,29.29 M,1.03,3.5 T USD,33.46,14.06 USD,+16.01%,0.73%,Technology services,Strong buy,Sell,NaN
4,"AMZNAmazon.com, Inc.",238.42 USD,−0.31%,32.83 M,0.82,2.55 T USD,33.68,7.08 USD,+51.70%,0.00%,Retail trade,Strong buy,Buy,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5095,CRNGCORENERGY INFRASTRUCTURE TR INCD,2.26 USD,0.00%,157,0.09,35.75 M USD,—,—,—,0.00%,Finance,—,Strong sell,NaN
5096,FEMYFemasys Inc.,0.6000 USD,−4.05%,483.89 K,0.32,35.7 M USD,—,−0.72 USD,+12.49%,0.00%,Electronic technology,Strong buy,Strong sell,NaN
5097,"KITTNauticus Robotics, Inc.",1.27 USD,+40.08%,33.27 M,8.80,35.63 M USD,—,−200.82 USD,+51.09%,0.00%,Producer manufacturing,—,Buy,NaN
5098,MYOMyomo Inc.,0.9228 USD,+1.15%,585.84 K,1.34,35.47 M USD,—,−0.29 USD,−25.54%,0.00%,Distribution services,Strong buy,Strong sell,NaN


In [1]:
import requests
import pandas as pd

url = "https://scanner.tradingview.com/america/scan"

headers = {
    "Content-Type": "application/json",
    "User-Agent": "Mozilla/5.0"
}

all_data = []
batch_size = 5000
start = 0

print("Fetching all stocks from TradingView America market...")

while True:
    payload = {
        "filter": [],
        "symbols": {"query": {"types": []}, "tickers": []},
        "columns": [
            "name",
            "close",
            "open",
            "volume",
            "market_cap_basic",
            "premarket_close",
            "premarket_open",
            "postmarket_close",
            "postmarket_open",
            "Recommend.All"
        ],
        "sort": {
            "sortBy": "market_cap_basic",
            "sortOrder": "desc"
        },
        "range": [start, start + batch_size]
    }
    
    res = requests.post(url, json=payload, headers=headers)
    data = res.json()["data"]
    
    if not data:
        break
    
    all_data.extend(data)
    print(f"Fetched batch: {len(data)} stocks | Total so far: {len(all_data)}")
    
    if len(data) < batch_size:
        break
    
    start += batch_size

# Convert all data to DataFrame
df = pd.DataFrame(
    [{**{"symbol": d["s"]}, **dict(zip(payload["columns"], d["d"]))}
     for d in all_data]
)

print(f"\n✓ Complete! Total stocks fetched: {len(df)}")
print(df.head())

Fetching all stocks from TradingView America market...
Fetched batch: 5000 stocks | Total so far: 5000
Fetched batch: 5000 stocks | Total so far: 10000
Fetched batch: 5000 stocks | Total so far: 15000
Fetched batch: 4300 stocks | Total so far: 19300

✓ Complete! Total stocks fetched: 19300
         symbol   name    close    open      volume  market_cap_basic  \
0   NASDAQ:NVDA   NVDA  191.435  191.27  85314236.0      4.651871e+12   
1  NASDAQ:GOOGL  GOOGL  333.126  336.06  12781783.0      4.022254e+12   
2   NASDAQ:GOOG   GOOG  333.310  336.61   7215279.0      4.019070e+12   
3   NASDAQ:AAPL   AAPL  254.840  257.65  16008854.0      3.745619e+12   
4   NASDAQ:MSFT   MSFT  478.840  483.21  10457024.0      3.558920e+12   

   premarket_close  premarket_open postmarket_close postmarket_open  \
0           191.28          191.54             None            None   
1           336.20          335.70             None            None   
2           336.48          336.44             None      